# Fetching data from SQL database

In [2]:
import mysql.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

conn = mysql.connector.connect(
    host='127.0.0.1',
    port=13306,
    user='root',
    password='secret',
    database='odb'
)

cursor = conn.cursor()

# Function for fetching data from database
def fetch_data_from_db():
    query = "SELECT * FROM terrorism"
    cursor.execute(query)
    result = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(result, columns=columns)
    return df


df = fetch_data_from_db()


#print(df.head())
#print(df.dtypes)
print(df.columns)

Index(['eventid', 'year', 'month', 'day', 'country', 'country_txt', 'region',
       'region_txt', 'city', 'success', 'suicide', 'attacktype',
       'attacktype_txt', 'target_type', 'target_type_txt', 'victim_nat',
       'victim_nat_txt', 'attacker_group', 'motive', 'weapon_type',
       'weapon_type_txt', 'fatalities', 'wounded', 'ransom', 'ransom_demanded',
       'ransom_paid'],
      dtype='object')


In [ ]:
country_counts = df['country_txt'].value_counts().sort_index().reset_index()
country_counts.columns = ['country', 'count']

# Create scatter plot
#fig = go.Figure(go.Scattergeo())

#fig.update_geos(projection_type='orthographic')
#fig.update_layout(height=400)


import pycountry

def country_to_iso(name):
    try:
        return pycountry.countries.lookup(name).alpha_3  # or use .alpha_2 for 2-letter codes
    except LookupError:
        return 2
    
country_counts['iso_code'] = country_counts['country'].apply(country_to_iso)


print(country_counts)

fig = px.choropleth(country_counts, locations='iso_code', color='count', hover_data=['country', 'count'])

fig.update_geos(projection_type='orthographic')
# Show the figure
fig.show()


                      country  count iso_code
0                 Afghanistan   2604      AFG
1                     Albania      1      ALB
2                     Algeria      4      DZA
3                      Angola      1      AGO
4                   Argentina      5      ARG
..                        ...    ...      ...
96              United States    103      USA
97                  Venezuela      7      VEN
98   West Bank and Gaza Strip     61        2
99                      Yemen    474      YEM
100                  Zimbabwe      1      ZWE

[101 rows x 3 columns]


In [50]:
# Get list of countries already in the DataFrame
present_countries = country_counts['country'].tolist()

# Prepare rows for missing countries
missing_rows = []


for c in pycountry.countries:
    print(c)
    if c.name not in present_countries:
        missing_rows.append({
            'country': c.name,
            'count': 0,
            'iso_code': c.alpha_3
        })

# Append missing countries
if missing_rows:
    country_counts = pd.concat([country_counts, pd.DataFrame(missing_rows)], ignore_index=True)

# Optional: sort alphabetically or by iso_code
country_counts = country_counts.sort_values(by='country').reset_index(drop=True)

print(country_counts)


fig = px.choropleth(country_counts, locations='iso_code', color='count', hover_data=['country', 'count'], color_continuous_scale = ["#fff5eb", "#fd8d3c", "#f03b20", "#bd0026", "#800026"])

fig.update_geos(projection_type='orthographic')
# Show the figure
fig.show()


Country(alpha_2='AW', alpha_3='ABW', flag='🇦🇼', name='Aruba', numeric='533')
Country(alpha_2='AF', alpha_3='AFG', flag='🇦🇫', name='Afghanistan', numeric='004', official_name='Islamic Republic of Afghanistan')
Country(alpha_2='AO', alpha_3='AGO', flag='🇦🇴', name='Angola', numeric='024', official_name='Republic of Angola')
Country(alpha_2='AI', alpha_3='AIA', flag='🇦🇮', name='Anguilla', numeric='660')
Country(alpha_2='AX', alpha_3='ALA', flag='🇦🇽', name='Åland Islands', numeric='248')
Country(alpha_2='AL', alpha_3='ALB', flag='🇦🇱', name='Albania', numeric='008', official_name='Republic of Albania')
Country(alpha_2='AD', alpha_3='AND', flag='🇦🇩', name='Andorra', numeric='020', official_name='Principality of Andorra')
Country(alpha_2='AE', alpha_3='ARE', flag='🇦🇪', name='United Arab Emirates', numeric='784')
Country(alpha_2='AR', alpha_3='ARG', flag='🇦🇷', name='Argentina', numeric='032', official_name='Argentine Republic')
Country(alpha_2='AM', alpha_3='ARM', flag='🇦🇲', name='Armenia', num